<a href="https://colab.research.google.com/github/bauer-san/CCRT-route-planner/blob/main/Route_Planner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ortools pandas googlemaps

  Preparing metadata (setup.py) ... done
  Created wheel for googlemaps: filename=googlemaps-4.10.0-py3-none-any.whl size=40714 sha256=aa3c8714673ebb119607ead24cbac783852f9bec6f10928de03493b0511e6d5e
  Stored in directory: /root/.cache/pip/wheels/4c/6a/a7/bbc6f5c200032025ee655deb5e163ce8594fa05e67d973aad6
Successfully built googlemaps


In [10]:
## THIS OPTIMIZATION WORKS AS INTENDED
from ortools.constraint_solver import routing_enums_pb2
from ortools.constraint_solver import pywrapcp
import googlemaps
from google.colab import userdata
from pprint import pprint
import urllib.parse

# Initialize Google Maps client (assuming API key is loaded from secrets in a previous cell)
gmaps = googlemaps.Client(key=userdata.get('GOOGLE_MAPS_API_KEY'))

# 1. DATA PREPARATION
def create_data_model(addresses, num_vehicles, depot_index=0):
    data = {}
    data['addresses'] = addresses
    data['num_vehicles'] = num_vehicles
    data['depot'] = depot_index

    num_addresses = len(addresses)
    full_distance_matrix = [[0 for _ in range(num_addresses)] for _ in range(num_addresses)]
    chunk_size = 10 # To respect the 100 elements (origin * destination) limit per API call

    print("Fetching distance matrix from Google Maps API in chunks...")

    try:
        for i in range(0, num_addresses, chunk_size):
            origin_chunk = addresses[i:i + chunk_size]
            for j in range(0, num_addresses, chunk_size):
                destination_chunk = addresses[j:j + chunk_size]

                matrix_result = gmaps.distance_matrix(
                    origins=origin_chunk,
                    destinations=destination_chunk,
                    mode="driving",
                    units="metric"
                )

                if matrix_result['status'] == 'OK':
                    for idx_origin, origin in enumerate(matrix_result['rows']):
                        for idx_dest, element in enumerate(origin['elements']):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest

                            if element['status'] == 'OK':
                                # Distance is in meters, store directly
                                full_distance_matrix[global_origin_index][global_destination_index] = element['distance']['value']
                            else:
                                # Placeholder for unreachable or error
                                full_distance_matrix[global_origin_index][global_destination_index] = 999999999 # Using a large number for meters
                else:
                    print(f"Error fetching distance matrix for chunk (origins {i}-{i+len(origin_chunk)-1}, destinations {j}-{j+len(destination_chunk)-1}): {matrix_result['status']}")
                    # If a chunk fails, we might want to fill with a large value or raise an error
                    # For simplicity, filling the failed chunk area with 999999999 for all elements (large number in meters)
                    for idx_origin in range(len(origin_chunk)):
                        for idx_dest in range(len(destination_chunk)):
                            global_origin_index = i + idx_origin
                            global_destination_index = j + idx_dest
                            full_distance_matrix[global_origin_index][global_destination_index] = 999999999

        data['distance_matrix'] = full_distance_matrix
        print("Distance matrix fetched and compiled successfully.")

    except Exception as e:
        print(f"An unexpected error occurred during API calls: {e}")
        # Fallback to a dummy matrix or raise an error
        data['distance_matrix'] = [[0 for _ in addresses] for _ in addresses] # Dummy matrix

    return data

# 2. THE SOLVER (The "Brain")
def solve_routing(data):
    manager = pywrapcp.RoutingIndexManager(len(data['distance_matrix']),
                                           data['num_vehicles'], data['depot'])
    routing = pywrapcp.RoutingModel(manager)

    def distance_callback(from_index, to_index):
        return data['distance_matrix'][manager.IndexToNode(from_index)][manager.IndexToNode(to_index)]

    transit_callback_index = routing.RegisterTransitCallback(distance_callback)
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)

    # Balancing load (max distance per team). Adjust this value based on typical route lengths.
    # The distance is in meters. So 5000000 means 5000 km in meters.
    routing.AddDimension(transit_callback_index, 0, 5000000, True, 'Distance') # Capacity now in meters

    # Add objective: Minimize the maximum distance traveled by any vehicle.
    distance_dimension = routing.GetDimensionOrDie('Distance')
    # Add a global span constraint to the distance dimension (optional, but good practice)
    # distance_dimension.SetGlobalSpanCostCoefficient(100) # This is for minimizing span, not max

    # Minimize the maximum distance traveled by any vehicle.
    # We need to find the maximum cumul value among all vehicles at their respective end nodes.
    # The objective is to minimize this maximum value.
    routing.SetArcCostEvaluatorOfAllVehicles(transit_callback_index)
    # Add a dimension for distance
    distance_dimension = routing.GetDimensionOrDie('Distance')
    distance_dimension.SetGlobalSpanCostCoefficient(100)

    search_params = pywrapcp.DefaultRoutingSearchParameters()
    search_params.first_solution_strategy = (
        routing_enums_pb2.FirstSolutionStrategy.PATH_CHEAPEST_ARC)

    return routing, manager, routing.SolveWithParameters(search_params)

# 3. THE FORMATTER (The "Output")
def get_readable_output(data, manager, routing, solution):
    results = {}
    for vehicle_id in range(data['num_vehicles']):
        index = routing.Start(vehicle_id)
        route_for_team = []
        route_distance = 0
        while not routing.IsEnd(index):
            node_index = manager.IndexToNode(index)
            route_for_team.append(data['addresses'][node_index])
            previous_index = index
            index = solution.Value(routing.NextVar(index))
            route_distance += routing.GetArcCostForVehicle(previous_index, index, vehicle_id)

        # Add the final return to depot
        route_for_team.append(data['addresses'][manager.IndexToNode(index)])
        results[f"Team {vehicle_id + 1}"] = {
            'route': route_for_team,
            'distance': route_distance / 1000 # Distance in the unit returned by callback (meters in this case)
        }
    return results

def print_final_manifests(route_results):
    for team, details in route_results.items():
        stops = details['route']
        distance = details['distance']
        print(f"\n=== {team.upper()} MANIFEST ===")
        print(f"Total Distance: {distance} meters") # Display in meters
        # Generate Google Maps Link
        encoded_waypoints = [urllib.parse.quote_plus(stop) for stop in stops[1:-1]] # URL encode waypoints
        waypoints_str = "|".join(encoded_waypoints)
        gmaps_url = f"https://www.google.com/maps/dir/?api=1&origin={urllib.parse.quote_plus(stops[0])}&destination={urllib.parse.quote_plus(stops[-1])}&waypoints={waypoints_str}"

        print(f"Digital Route: {gmaps_url}")
        print(f"{'Stop #':<8} | {'Address'}")
        print("-" * 40)
        for i, addr in enumerate(stops):
            label = "START" if i == 0 else "END" if i == len(stops)-1 else f"Stop {i}"
            print(f"{label:<8} | {addr}")

# 4. EXECUTION
if __name__ == '__main__':
    # Example usage with your addresses
    original_sample_addresses = [
        "49 Clinton St, Pontiac, MI, USA",
        "53 Waldo St, Pontiac, MI, USA",
        "3000 Winston Dr, Apt 3108, Pontiac, MI, USA",
        "60 Mechanic St, Pontiac, MI, USA"
    ]
    sample_addresses = [
    "6 N Saginaw St, Pontiac, MI, USA",
    "22 N Saginaw St, Pontiac, MI, USA",
    "51 N Saginaw St, Pontiac, MI, USA",
    "67 N Saginaw St, Pontiac, MI, USA",
    "108 N Saginaw St, Pontiac, MI, USA",
    "180 N Saginaw St, Pontiac, MI, USA",
    "20 Franklin Rd, Pontiac, MI, USA",
    "18 W Huron St, Pontiac, MI, USA",
    "47450 Woodward Ave, Pontiac, MI, USA",
    "44405 Woodward Ave, Pontiac, MI, USA",
    "60 E Pike St, Pontiac, MI, USA",
    "51000 Woodward Ave, Pontiac, MI, USA",
    "1275 N Perry St, Pontiac, MI, USA",
    "1051 Arlene Ave, Pontiac, MI, USA",
    "154 Lake St, Pontiac, MI, USA",
    "312 S Edith St, Pontiac, MI, USA",
    "858 Amanda Ln, Pontiac, MI, USA",
    "256 Cedardale Ave, Pontiac, MI, USA",
    "520 Linda Vista Dr, Pontiac, MI, USA",
    "160 Chippewa Rd, Pontiac, MI, USA",
    "215 Oak Ridge Dr, Pontiac, MI, USA",
    "738 Arusha Dr, Pontiac, MI, USA",
    "20 Park Pl, Pontiac, MI, USA",
    "180 W Rutgers Ave, Pontiac, MI, USA",
    "635 E Tennyson Ave, Pontiac, MI, USA"
]
    num_teams = 3

    main_data = create_data_model(sample_addresses, num_teams)

    if 'distance_matrix' in main_data and main_data['distance_matrix']:
        routing_model, routing_manager, solution_obj = solve_routing(main_data)

        if solution_obj:
            readable_routes = get_readable_output(main_data, routing_manager, routing_model, solution_obj)
            print_final_manifests(readable_routes)
        else:
            print("No solution found!")
    else:
        print("Could not create data model due to missing distance matrix. Check API key and network connection.")

Fetching distance matrix from Google Maps API in chunks...
Distance matrix fetched and compiled successfully.

=== TEAM 1 MANIFEST ===
Total Distance: 12.34 meters
Digital Route: https://www.google.com/maps/dir/?api=1&origin=6+N+Saginaw+St%2C+Pontiac%2C+MI%2C+USA&destination=6+N+Saginaw+St%2C+Pontiac%2C+MI%2C+USA&waypoints=51000+Woodward+Ave%2C+Pontiac%2C+MI%2C+USA|20+Franklin+Rd%2C+Pontiac%2C+MI%2C+USA|215+Oak+Ridge+Dr%2C+Pontiac%2C+MI%2C+USA|256+Cedardale+Ave%2C+Pontiac%2C+MI%2C+USA|44405+Woodward+Ave%2C+Pontiac%2C+MI%2C+USA|312+S+Edith+St%2C+Pontiac%2C+MI%2C+USA|20+Park+Pl%2C+Pontiac%2C+MI%2C+USA|47450+Woodward+Ave%2C+Pontiac%2C+MI%2C+USA|60+E+Pike+St%2C+Pontiac%2C+MI%2C+USA|22+N+Saginaw+St%2C+Pontiac%2C+MI%2C+USA
Stop #   | Address
----------------------------------------
START    | 6 N Saginaw St, Pontiac, MI, USA
Stop 1   | 51000 Woodward Ave, Pontiac, MI, USA
Stop 2   | 20 Franklin Rd, Pontiac, MI, USA
Stop 3   | 215 Oak Ridge Dr, Pontiac, MI, USA
Stop 4   | 256 Cedardale Ave, P